In [1]:
import pandas as pd
from tqdm import tqdm
import torch
import torch.nn as nn
import time
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import numpy as np

import random

def set_seed(seed):
    """Sets the seed for reproducibility."""
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)  # if using multi-GPU.
    np.random.seed(seed)  # Numpy module.
    random.seed(seed)  # Python's random module.
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.enabled = False

# Example usage:
seed = 1000
set_seed(seed)

In [40]:
# Try with a subset of specific types of restaurants
df_train = pd.read_csv('./data/hotdogs-train.csv')
df_test = pd.read_csv('./data/hotdogs-test.csv')


Use the below to downsample a large dataset

In [29]:
# Skip this if not using df_true
# df_true is the big dataset (10mil samples)
# df_true = pd.read_csv('./data/midwest/midwest-big-train.csv')

def proportional_sample(df, col_name, sample_size):
    value_counts = df[col_name].value_counts(normalize=True)
    sampled_counts = (value_counts * sample_size).round().astype(int)

    sampled_dfs = []
    for value, count in sampled_counts.items():
        subset = df[df[col_name] == value].sample(n=min(count, len(df[df[col_name] == value])), replace=False) #sample without replacement, take the minimum of the count or the amount of that value in the df.
        sampled_dfs.append(subset)

    sampled_df = pd.concat(sampled_dfs)
    return sampled_df

## Take a proportional slice of the text to practice with
sample_size = 1000000
df = proportional_sample(df_true, 'rating', sample_size)

NameError: name 'df_true' is not defined

In [4]:
df.head()

,rating,text
0,5,Love this place!
1,5,Good food great ice cream. Plus garden supply...
2,5,Amazing Chicago comfort food with awesome serv...
3,5,"Love this place! Hots dogs, polish and shakes ..."
4,5,Love their beef w/hot peppers mmm


In [37]:
import re
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

def preprocess_text(text):
    # Improve preprocessing dramatically improves accuracy
    text = str(text)
    
    # This appears in some reviews
    text = text.replace('(Translated by Google)', ' ')
    text = text.replace('\n', ' ')

    # Remove non letter or number entries
    # BERT does OK with numbers is seems
    text = re.sub(r'[^\w\s]', '', text)
    
    # Convert to lowercase
    text = text.lower()

    return text

# Process the text for better classification
df_train['text'] = df_train['text'].apply(preprocess_text)
df_test['text'] = df_test['text'].apply(preprocess_text)

In [41]:
# Turn ratings into +/-/neutral and do a bit of light processing to the text
# Careful to only run this once

def pnn(n):
    if n in {1, 2}: return 0
    elif n in {3}: return 1
    else: return 2

def minus1(n):
    return n-1
    
df_train['pnn'] = df_train['rating'].apply(pnn)
df_test['pnn'] = df_test['rating'].apply(pnn)

df_train['rating'] = df_train['rating'].apply(minus1)
df_test['rating'] = df_test['rating'].apply(minus1)

In [42]:
print('Train set sample')
print(df_train.sample(8))
print()

print('Test set sample')
print(df_test.sample(8))

Train set sample
       rating                                               text  pnn
31685       4                                  Always great food    2
46617       3  The burgers are very creative but obviously no...    2
39862       3  Have you ever noticed that the food is never a...    2
2611        4  Vegan and veggie options!!  Salads galore, fru...    2
52845       2                 Just got root beer float very good    1
16340       4                                         Great food    2
11375       4           Good coneys. Good price.  Great service.    2
10912       4     Great food, Great service, lovely environment.    2

Test set sample
       rating                                               text  pnn
2254        4      (Translated by Google) Me!\n\n(Original)\nYo!    2
8313        4                              Mexican food is great    2
13778       0  I placed an order at 6:44 pm for two Chicago s...    0
821         4  Food is alway freash and so delicious. Pr

Instantiate the bert model. Tried both base and large, no real difference in acc

In [26]:
from transformers import BertTokenizer, BertModel, BertForSequenceClassification
from transformers import AutoModelForSequenceClassification, TFAutoModelForSequenceClassification
from transformers import AutoTokenizer, AutoConfig

model_name = f"cardiffnlp/twitter-roberta-base-sentiment-latest"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load the tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(model_name)
config = AutoConfig.from_pretrained(model_name)

# PT
model = AutoModelForSequenceClassification.from_pretrained(model_name).to(device)

#print(model)
print('Device: ', device)

Some weights of the model checkpoint at cardiffnlp/twitter-roberta-base-sentiment-latest were not used when initializing RobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


Device:  cuda


These cells can be used to predict one review at a time, or enter your own custom text

In [25]:
def tokenize_text(text):
    inputs = tokenizer(text, padding=True, truncation=True, return_tensors="pt")
    return inputs

In [117]:
def predict_sentiment(text):
    inputs = tokenize_text(text)
    inputs = {key: val.to(device) for key, val in inputs.items()} #move the inputs to the correct device.
    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits
        probabilities = torch.softmax(logits, dim=1)
        predicted_class = torch.argmax(probabilities, dim=1).item()
    return predicted_class

In [ ]:
# Simple test cycle of the untrained model
review = 'The best chicken ever'
label = predict_sentiment(review)
meaning = ['negative', 'neutral/average', 'good']

print('Review: ', review)
print(f'Predicted rating: {label} {meaning[label]}')

Run this so that the training autogenerates a plot after finishing


In [17]:
import datetime
import matplotlib.pyplot as plt

def plot_lists_with_colors(list1, list2, label1="List 1", label2="List 2", color1="blue", color2="red"):
    # Ensure lists have equal length (or adjust as needed)
    min_len = min(len(list1), len(list2))
    x_values = np.arange(min_len)

    # Create the plot
    plt.figure(figsize=(10, 6))  # Adjust figure size if needed

    # Plot List 1
    plt.plot(x_values, list1[:min_len], color=color1, label=label1, linestyle='none', marker='o')
    plt.plot(x_values, list2[:min_len], color=color2, label=label2, linestyle='none', marker='o')
    current_time = datetime.datetime.now()
    
    # Add Labels and Title
    plt.xlabel("Epoch") 
    plt.ylabel("Accuracy (%)")
    plt.title(f"Bert - Midwest data ({current_time.strftime("%Y-%m-%d %H:%m")})")

    plt.legend()
    plt.grid(True)

    file_name = f'./sshots/model-{current_time.strftime("%Y-%m-%d--%H%m")}.png'
    plt.savefig(file_name)
    
    # Show the Plot
    plt.show()

## Training the BERT model begins here

In [18]:
from torch.utils.data import DataLoader, Dataset
from sklearn.model_selection import train_test_split

class SentimentDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=128): #add max_length.
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length #store max length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        label = self.labels[idx]
        
        inputs = self.tokenizer(
            text,
            padding="max_length", #important
            truncation=True, #important
            max_length=self.max_length, #important
            return_tensors="pt")
        
        input_ids = inputs['input_ids'].flatten()
        attention_mask = inputs['attention_mask'].flatten()
        label_tensor = torch.tensor(label)

        return {
            'input_ids': input_ids,
            'attention_mask': attention_mask,
            'labels': label_tensor
        }

In [19]:
from sklearn.utils import class_weight

# Ratings are not evenly distributed, this creates class weights
def calculate_class_weights(labels):
    class_weights = class_weight.compute_class_weight('balanced', classes=np.unique(labels), y=labels)
    return torch.tensor(class_weights, dtype=torch.float)

In [43]:
# Export the text and ratings to a list, this part is necessary otherwise pandas keeps the index
X_train = df_train['text'].values.tolist()
y_train = df_train['pnn'].values.tolist()
X_test = df_test['text'].values.tolist()
y_test = df_test['pnn'].values.tolist()

# The batch size seems to need to be pretty small, anything more than 32 crashed my 3070 with 8GB VRAM
# monitor RAM use in the terminal with "watch -n5 nvidia-smi"
# Other parameters for the training are set here as well
batch_size = 64
max_length = 128
learning_rate = 2e-5
weight_decay = .01

## FREEZE the base BERT parameters and only train the classification layer
for param in model.roberta.parameters():
    param.requires_grad = False

train_dataset = SentimentDataset(X_train, y_train, tokenizer, max_length=max_length)
test_dataset = SentimentDataset(X_test, y_test, tokenizer, max_length=max_length)

train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_dataloader = DataLoader(test_dataset, batch_size=batch_size)

# Adjust learning rate and weight decay
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)

# Calculate class weights only after the split
class_weights = calculate_class_weights(y_train).to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights)

print('Class weights:', class_weights)

Class weights: tensor([3.9634, 4.1611, 0.3988], device='cuda:0')


In [44]:
## Set this very high and use an early exit
epochs = 2

accs = [[], []] #0 = Train, 1 = Test

for epoch in range(epochs):
    #Train cycle
    model.train()
    
    train_loss = 0
    all_predicted_labels, all_true_labels = [], []
    
    for batch in tqdm(train_dataloader, desc=f"Epoch {epoch + 1}/{epochs}"):
        optimizer.zero_grad()

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)
        
        outputs = model(input_ids, attention_mask=attention_mask, labels=labels)

        ## used for calculating accuracy
        _, predicted_labels = torch.max(outputs.logits, dim=1)
        all_predicted_labels.extend(predicted_labels.cpu().numpy())
        all_true_labels.extend(labels.cpu().numpy())

        ## Training
        loss = outputs.loss
        train_loss += loss.item()
        
        loss.backward()
        
        optimizer.step()

    acc = accuracy_score(all_true_labels, all_predicted_labels)
    accs[0].append(100*acc)
    tr_l = train_loss / len(train_dataloader)
    
    # Test cycle
    print('Testing...', end='\r')
    model.eval()
    
    test_loss = 0
    all_predicted_labels, all_true_labels = [], []
    
    with torch.no_grad():
        for batch in test_dataloader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)
            
            outputs = model(input_ids, attention_mask=attention_mask)
            
            _, predicted_labels = torch.max(outputs.logits, dim=1)
            all_predicted_labels.extend(predicted_labels.cpu().numpy())
            all_true_labels.extend(labels.cpu().numpy())

    acc = accuracy_score(all_true_labels, all_predicted_labels)
    accs[1].append(100*acc)
    
    print(f'Train loss: {tr_l:.5f}, acc {accs[0][epoch]:.2f}% | Test acc: {accs[1][epoch]:.2f}%')

# Plot the accuracies on completion
plot_lists_with_colors(accs[0], accs[1], label1="Train accuracy", label2="Test accuracy")

Epoch 1/2:   9%|██▉                            | 89/939 [00:15<02:24,  5.89it/s]


KeyboardInterrupt: 

Save the model

In [38]:
## Save the model here
torch.save(model.state_dict(), f'./saved-models/model-bert-hotdogs-3-good.pth')

## Testing saved models below here

In [167]:
model = torch.load('./saved-models/model-hotdogs.pth').to(device)

/tmp/ipykernel_662689/3911888007.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model = torch.load('./saved-models/model-hotdogs.pth').to(device)


AttributeError: Can't get attribute 'LSTMModel' on <module '__main__'>

In [45]:
# Evaluate loop only
model.eval()
    
test_loss = 0
all_predicted_labels, all_true_labels = [], []

with torch.no_grad():
    for batch in tqdm(test_dataloader):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)
        
        outputs = model(input_ids, attention_mask=attention_mask)
        
        _, predicted_labels = torch.max(outputs.logits, dim=1)
        all_predicted_labels.extend(predicted_labels.cpu().numpy())
        all_true_labels.extend(labels.cpu().numpy())

print(f'Accuracy: {100*accuracy_score(all_true_labels, all_predicted_labels):.2f}%')

100%|█████████████████████████████████████████| 235/235 [00:38<00:00,  6.17it/s]

Accuracy: 89.52%


## Suggestions

 Set aside cross-validation set - 
 
 Total random seed pytorch level torch.manual_seed() - DONE
  
 Early exit with high number of epochs
 
 Play with groupings of ratings (1--5, pos/neg only -- create neutral based on logits) - TRIED WITH 0--4

 Learning rates: 5e-5, 4e-5, 3e-5, and 2e-5, .0001. Learning rate scheduler

 Investigate BERT classifier more

 Try different BERT model -- roberta? https://huggingface.co/AnkitAI/reviews-roberta-base-sentiment-analysis